In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

In [ ]:
train_data = pd.read_csv("../dataset/train_metadata.csv")
val_data = pd.read_csv("../dataset/val_metadata.csv")
test_data = pd.read_csv("../dataset/test_metadata.csv")

In [ ]:
IMAGE_SIZE = 224
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image = cv2.resize(image,(IMAGE_SIZE, IMAGE_SIZE))
    image = image / 255.0
    return image.astype(np.float32)

In [ ]:
sample_image = preprocess_image(
    train_data.iloc[0]["image_path"]
)


sample_image.shape

In [ ]:
def create_dataset(data):
    image_paths = data["image_path"].values
    labels = data["label"].values
    dataset = tf.data.Dataset.from_tensor_slices((image_paths,labels))
    def load_image(path,label):
        image = tf.numpy_function(preprocess_image,[path],tf.float32)
        image.set_shape((224,224,3))
        return image,label
    dataset = dataset.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

In [ ]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)

In [ ]:
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(0.1)

])

In [ ]:
def apply_augmentation(image,label):
    image = data_augmentation(image)
    return image,label
train_dataset = train_dataset.map(apply_augmentation,num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
BATCH_SIZE = 32
train_dataset = (train_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset = (val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_dataset = (test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [ ]:
class_weights = compute_class_weight(class_weight="balanced",classes=np.unique(train_data["label"]),y=train_data["label"])
class_weights

In [ ]:
class_weights = dict(enumerate(class_weights))
class_weights

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models


base_model = ResNet50(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False

resnet50_model = models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])


resnet50_model.summary()

In [ ]:
resnet50_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)

In [ ]:
history_eff = resnet50_model.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

In [ ]:
train_loss, train_accuracy = resnet50_model.evaluate(train_dataset)
val_loss, val_accuracy = resnet50_model.evaluate(val_dataset)
test_loss, test_accuracy = resnet50_model.evaluate(test_dataset)

In [ ]:
resnet50_model.save("../models/resnet50_model.keras")

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models


base_model = ResNet50(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False

resnet50_model_sgd= models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])


resnet50_model.summary()

In [ ]:
resnet50_model_sgd.compile(

    optimizer=tf.keras.optimizers.SGD(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_res_sgd = resnet50_model_sgd.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

In [ ]:
train_loss, train_accuracy = resnet50_model_sgd.evaluate(train_dataset)
val_loss, val_accuracy = resnet50_model_sgd.evaluate(val_dataset)
test_loss, test_accuracy = resnet50_model_sgd.evaluate(test_dataset)

In [ ]:
resnet50_model_sgd.save("../models/resnet50_model_sgd.keras")

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models


base_model = ResNet50(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False

resnet50_model_RMSprop = models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])


resnet50_model_RMSprop.summary()

In [ ]:
resnet50_model_RMSprop.compile(

    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_res_RMSprop = resnet50_model_RMSprop.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

In [ ]:
train_loss, train_accuracy = resnet50_model_RMSprop.evaluate(train_dataset)
val_loss, val_accuracy = resnet50_model_RMSprop.evaluate(val_dataset)
test_loss, test_accuracy = resnet50_model_RMSprop.evaluate(test_dataset)

In [ ]:
resnet50_model_RMSprop.save("../models/resnet50_model_RMSprop.keras")

In [ ]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)
 
train_dataset = train_dataset.map(
    apply_augmentation,
    num_parallel_calls=tf.data.AUTOTUNE
)
 
BATCH_SIZE = 64
 
train_dataset = (
    train_dataset
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models


base_model = ResNet50(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False

resnet50_model_64= models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])


resnet50_model_64.summary()

In [ ]:
resnet50_model_64.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_res_64= resnet50_model_64.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

In [ ]:
train_loss, train_accuracy = resnet50_model_64.evaluate(train_dataset)
val_loss, val_accuracy = resnet50_model_64.evaluate(val_dataset)
test_loss, test_accuracy = resnet50_model_64.evaluate(test_dataset)

In [ ]:
resnet50_model_64.save("../models/resnet50_model_64.keras")

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Load MobileNetV2

base_model = ResNet50(

    weights="imagenet",

    include_top=False,

    input_shape=(224,224,3)

)

# Fine-Tuning

base_model.trainable = True

# Freeze all layers except the last 30

for layer in base_model.layers[:-30]:

    layer.trainable = False

# Build Model

res_ft = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.5),

    layers.Dense(
        7,
        activation="softmax"
    )

])

res_ft.summary()

In [ ]:
res_ft.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_res_ft = res_ft.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

In [ ]:
train_loss, train_accuracy = res_ft.evaluate(train_dataset)
val_loss, val_accuracy = res_ft.evaluate(val_dataset)
test_loss, test_accuracy = res_ft.evaluate(test_dataset)

In [ ]:
res_ft.save("../models/res_ft.keras")

In [ ]:
import keras_tuner as kt
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam, RMSprop

num_classes = 7

def build_model(hp):

    base_model = ResNet50(

        weights="imagenet",

        include_top=False,

        input_shape=(224,224,3)

    )

    base_model.trainable = False

    res_model = models.Sequential([

        base_model,

        layers.GlobalAveragePooling2D(),

        layers.Dense(

            units=hp.Choice(

                "dense_units",

                [128,256,512]

            ),

            activation="relu"

        ),

        layers.Dropout(

            hp.Choice(

                "dropout",

                [0.3,0.5,0.6]

            )

        ),

        layers.Dense(

            num_classes,

            activation="softmax"

        )

    ])

    learning_rate = hp.Choice(

        "learning_rate",

        [1e-3,1e-4,1e-5]

    )

    optimizer = hp.Choice(

        "optimizer",

        ["adam","rmsprop"]

    )

    if optimizer == "adam":

        opt = Adam(

            learning_rate=learning_rate

        )

    else:

        opt = RMSprop(

            learning_rate=learning_rate

        )

    res_model.compile(

        optimizer=opt,

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]

    )

    return res_model

In [ ]:
tuner = kt.RandomSearch(

    res_model,

    objective="val_accuracy",

    max_trials=3,

    directory="resnet_tuner",

    project_name="mobilenet_hyperparameter"

)

In [ ]:
tuner.search(

    train_dataset,

    validation_data=val_dataset,

    epochs=5,

    class_weight=class_weights

)

In [ ]:
best_eff_net = tuner.get_best_models(1)[0]

In [ ]:
best_hps = tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)